# Notebook 2 — Clasificación SVC (BUY/SELL)

**Ernesto Investing AI — Integración (iDeSo)**

Este notebook lee los datos OHLCV + indicadores técnicos guardados por el
**Notebook 1** en la colección `precios_ohlcv`, entrena un **Support Vector
Classifier (SVC)** con `GridSearchCV` para predecir si el precio de cierre
sube o baja al día siguiente, y guarda:

- La **señal actual** (BUY/SELL) de cada ticker en la colección `predicciones`.
- Las **métricas del modelo** de cada ticker en la colección `metricas_modelos`.

Estas dos colecciones son las que luego expone el **Notebook 3** (API FastAPI)
en los endpoints `/api/svc/{ticker}`.

**Tickers:** FSM, VOLCABC1.LM, ABX.TO, BVN, BHP

**Nota sobre data leakage:** todos los features de un día `t` se calculan
únicamente con información disponible hasta `t` (indicadores rolling/ewm ya
calculados en el Notebook 1). El target es la dirección del precio en `t+1`.
El split train/test es **temporal** (sin mezclar) y el escalado
(`StandardScaler`) se ajusta **solo con el train**.


## 1. Instalación de librerías

In [13]:
# pymongo con soporte SRV para el URI de Atlas.
# scikit-learn, pandas y numpy ya vienen preinstalados en Colab.
!pip install -q "pymongo[srv]"


## 2. Conexión a MongoDB Atlas

In [15]:
from google.colab import userdata
from pymongo import MongoClient
from pymongo.server_api import ServerApi

# Leemos el URI desde los Secrets de Colab (nunca lo escribimos en texto plano)
MONGO_URI = userdata.get('MONGO_URI')

cliente = MongoClient(MONGO_URI, server_api=ServerApi('1'))

# Verificamos que la conexión funcione
try:
    cliente.admin.command('ping')
    print("✅ Conexión exitosa a MongoDB Atlas")
except Exception as e:
    print(f"❌ Error de conexión: {e}")

# Base de datos y colecciones
db = cliente['ernesto_investing_ai']
col_precios = db['precios_ohlcv']
col_predicciones = db['predicciones']
col_metricas = db['metricas_modelos']

print('Documentos en precios_ohlcv:', col_precios.count_documents({}))


✅ Conexión exitosa a MongoDB Atlas
Documentos en precios_ohlcv: 1252


## 3. Lectura de datos desde MongoDB

Leemos `precios_ohlcv` (poblada por el Notebook 1) y armamos un DataFrame
por ticker, ordenado cronológicamente.

In [16]:
import pandas as pd

TICKERS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']

datos_por_ticker = {}

for ticker in TICKERS:
    cursor = col_precios.find({'ticker': ticker}).sort('fecha', 1)
    df = pd.DataFrame(list(cursor))

    if df.empty:
        print(f"  ⚠️ Sin datos en MongoDB para {ticker}, se omite.")
        continue

    df['fecha'] = pd.to_datetime(df['fecha'])
    df = df.sort_values('fecha').reset_index(drop=True)
    datos_por_ticker[ticker] = df
    print(f"  ✅ {ticker}: {len(df)} filas leídas ({df['fecha'].min().date()} → {df['fecha'].max().date()})")

print(f"\nTickers con datos disponibles: {list(datos_por_ticker.keys())}")


  ✅ FSM: 251 filas leídas (2025-07-03 → 2026-07-02)
  ✅ VOLCABC1.LM: 247 filas leídas (2025-07-03 → 2026-07-02)
  ✅ ABX.TO: 252 filas leídas (2025-07-03 → 2026-07-03)
  ✅ BVN: 251 filas leídas (2025-07-03 → 2026-07-02)
  ✅ BHP: 251 filas leídas (2025-07-03 → 2026-07-02)

Tickers con datos disponibles: ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']


## 4. Ingeniería de features y target

**Features:** usamos los indicadores ya calculados en el Notebook 1
(`SMA_20`, `SMA_50`, `EMA_12`, `EMA_26`, `RSI_14`) y agregamos unos derivados,
todos calculados con datos hasta el día `t` (sin mirar el futuro):

- `MACD` = EMA_12 − EMA_26
- `dist_sma20` = Close / SMA_20 − 1 (qué tan lejos está el precio de su media de 20 días)
- `dist_sma50` = Close / SMA_50 − 1
- `retorno_1d` = variación porcentual del cierre respecto al día anterior
- `volatilidad_5d` = desviación estándar rolling de los retornos (5 días)

**Target:** `1` = BUY si el `Close` de **mañana** es mayor al de **hoy**,
`0` = SELL en caso contrario. Se calcula con `shift(-1)` y por eso la
**última fila de cada ticker se descarta** (no conocemos el cierre del día
siguiente todavía).

In [17]:
import numpy as np

COLUMNAS_FEATURES = [
    'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26', 'RSI_14',
    'MACD', 'dist_sma20', 'dist_sma50', 'retorno_1d', 'volatilidad_5d',
]

def calcular_features_y_target(df):
    """Genera features derivados y el target binario BUY/SELL.
    Todo lo que se usa como feature en la fila t se calcula con
    información disponible hasta t; el target mira t+1."""
    df = df.copy()

    df['MACD'] = df['EMA_12'] - df['EMA_26']
    df['dist_sma20'] = df['Close'] / df['SMA_20'] - 1
    df['dist_sma50'] = df['Close'] / df['SMA_50'] - 1
    df['retorno_1d'] = df['Close'].pct_change()
    df['volatilidad_5d'] = df['retorno_1d'].rolling(window=5).std()

    # Target: 1 (BUY) si el cierre de MAÑANA es mayor al de HOY
    df['target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

    # La última fila no tiene "mañana" conocido -> no sirve para entrenar
    df = df.iloc[:-1]

    # Descartamos filas sin historial suficiente para los indicadores
    # (por ejemplo, SMA_50 necesita 50 días previos)
    df = df.dropna(subset=COLUMNAS_FEATURES + ['target']).reset_index(drop=True)

    return df

datasets = {}
for ticker, df in datos_por_ticker.items():
    df_features = calcular_features_y_target(df)
    datasets[ticker] = df_features
    print(f"  {ticker}: {len(df_features)} filas utilizables tras calcular features/target")


  FSM: 201 filas utilizables tras calcular features/target
  VOLCABC1.LM: 197 filas utilizables tras calcular features/target
  ABX.TO: 202 filas utilizables tras calcular features/target
  BVN: 201 filas utilizables tras calcular features/target
  BHP: 201 filas utilizables tras calcular features/target


## 5. Split temporal 80/20 y normalización

El split respeta el orden cronológico: el 80% más antiguo es train, el 20%
más reciente es test. **No se mezcla (`shuffle=False`)** para no filtrar
información futura hacia el pasado. El `StandardScaler` se ajusta **solo
con el train** y luego se aplica igual al test.

In [18]:
from sklearn.preprocessing import StandardScaler

PORCENTAJE_TRAIN = 0.8

splits = {}

for ticker, df in datasets.items():
    n = len(df)
    corte = int(n * PORCENTAJE_TRAIN)

    if corte < 10 or (n - corte) < 5:
        print(f"  ⚠️ {ticker}: muy pocos datos ({n} filas) para un split confiable, se omite.")
        continue

    train_df = df.iloc[:corte]
    test_df = df.iloc[corte:]

    X_train_raw = train_df[COLUMNAS_FEATURES].values
    X_test_raw = test_df[COLUMNAS_FEATURES].values
    y_train = train_df['target'].values
    y_test = test_df['target'].values

    # El scaler se ajusta SOLO con train, y se aplica igual a test (sin
    # volver a ajustarlo) para no filtrar estadísticas del futuro.
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    splits[ticker] = {
        'X_train': X_train, 'X_test': X_test,
        'y_train': y_train, 'y_test': y_test,
        'scaler': scaler,
        'train_df': train_df, 'test_df': test_df,
        'df_completo': df,
    }
    print(f"  ✅ {ticker}: train={len(train_df)}  test={len(test_df)}")


  ✅ FSM: train=160  test=41
  ✅ VOLCABC1.LM: train=157  test=40
  ✅ ABX.TO: train=161  test=41
  ✅ BVN: train=160  test=41
  ✅ BHP: train=160  test=41


## 6. Entrenamiento SVC con GridSearchCV

Usamos `TimeSeriesSplit` como estrategia de validación cruzada dentro del
`GridSearchCV` (en vez del `KFold` por defecto, que mezcla el orden). Así,
incluso durante la búsqueda de hiperparámetros, cada fold de validación es
siempre posterior en el tiempo a su fold de entrenamiento.

In [19]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

GRID_PARAMS = {
    'kernel': ['linear', 'rbf'],
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto'],
}

modelos_entrenados = {}

for ticker, data in splits.items():
    print(f"Entrenando SVC para {ticker}...")

    # 5 folds respetando el orden temporal (evita leakage en el CV)
    cv_temporal = TimeSeriesSplit(n_splits=5)

    grid = GridSearchCV(
        estimator=SVC(probability=True, random_state=42),
        param_grid=GRID_PARAMS,
        cv=cv_temporal,
        scoring='f1',
        n_jobs=-1,
    )
    grid.fit(data['X_train'], data['y_train'])

    modelos_entrenados[ticker] = grid.best_estimator_
    print(f"  ✅ Mejores hiperparámetros: {grid.best_params_}  (F1 CV = {grid.best_score_:.4f})")


Entrenando SVC para FSM...
  ✅ Mejores hiperparámetros: {'C': 0.1, 'gamma': 'scale', 'kernel': 'rbf'}  (F1 CV = 0.6830)
Entrenando SVC para VOLCABC1.LM...
  ✅ Mejores hiperparámetros: {'C': 100, 'gamma': 'auto', 'kernel': 'rbf'}  (F1 CV = 0.5452)
Entrenando SVC para ABX.TO...
  ✅ Mejores hiperparámetros: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}  (F1 CV = 0.7286)
Entrenando SVC para BVN...
  ✅ Mejores hiperparámetros: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}  (F1 CV = 0.7390)
Entrenando SVC para BHP...
  ✅ Mejores hiperparámetros: {'C': 100, 'gamma': 'scale', 'kernel': 'linear'}  (F1 CV = 0.5285)


## 7. Evaluación: accuracy, precision, recall, F1 y matriz de confusión

In [20]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
)

metricas_por_ticker = {}

for ticker, modelo in modelos_entrenados.items():
    data = splits[ticker]
    y_pred = modelo.predict(data['X_test'])

    acc = accuracy_score(data['y_test'], y_pred)
    prec = precision_score(data['y_test'], y_pred, zero_division=0)
    rec = recall_score(data['y_test'], y_pred, zero_division=0)
    f1 = f1_score(data['y_test'], y_pred, zero_division=0)
    matriz = confusion_matrix(data['y_test'], y_pred).tolist()

    metricas_por_ticker[ticker] = {
        'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1,
        'matriz_confusion': matriz,
    }

    print(f"--- {ticker} ---")
    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1:        {f1:.4f}")
    print(f"  Matriz de confusión (filas=real, cols=predicho):\n{matriz}\n")


--- FSM ---
  Accuracy:  0.4634
  Precision: 0.4634
  Recall:    1.0000
  F1:        0.6333
  Matriz de confusión (filas=real, cols=predicho):
[[0, 22], [0, 19]]

--- VOLCABC1.LM ---
  Accuracy:  0.4250
  Precision: 0.4783
  Recall:    0.5000
  F1:        0.4889
  Matriz de confusión (filas=real, cols=predicho):
[[6, 12], [11, 11]]

--- ABX.TO ---
  Accuracy:  0.5610
  Precision: 0.5484
  Recall:    0.8095
  F1:        0.6538
  Matriz de confusión (filas=real, cols=predicho):
[[6, 14], [4, 17]]

--- BVN ---
  Accuracy:  0.5366
  Precision: 0.5000
  Recall:    0.6842
  F1:        0.5778
  Matriz de confusión (filas=real, cols=predicho):
[[9, 13], [6, 13]]

--- BHP ---
  Accuracy:  0.4878
  Precision: 0.5500
  Recall:    0.4783
  F1:        0.5116
  Matriz de confusión (filas=real, cols=predicho):
[[9, 9], [12, 11]]



## 8. Señal actual por ticker

Usamos la fila más reciente de cada ticker (la de hoy, con datos ya
conocidos) para predecir la dirección de mañana: esa es la "señal actual"
que consumirá el endpoint `/api/svc/{ticker}` del Notebook 3.

In [21]:
señales_actuales = {}

for ticker, modelo in modelos_entrenados.items():
    df_completo = datos_por_ticker[ticker].copy()

    # Recalculamos los features (incluida la última fila, que en el paso 4
    # se había descartado por no tener target conocido) para poder predecir
    # la señal de HOY sin necesidad de conocer el cierre de mañana.
    df_con_features = df_completo.copy()
    df_con_features['MACD'] = df_con_features['EMA_12'] - df_con_features['EMA_26']
    df_con_features['dist_sma20'] = df_con_features['Close'] / df_con_features['SMA_20'] - 1
    df_con_features['dist_sma50'] = df_con_features['Close'] / df_con_features['SMA_50'] - 1
    df_con_features['retorno_1d'] = df_con_features['Close'].pct_change()
    df_con_features['volatilidad_5d'] = df_con_features['retorno_1d'].rolling(window=5).std()
    df_con_features = df_con_features.dropna(subset=COLUMNAS_FEATURES).reset_index(drop=True)

    ultima_fila = df_con_features.iloc[[-1]]
    X_ultima = splits[ticker]['scaler'].transform(ultima_fila[COLUMNAS_FEATURES].values)

    prediccion = modelo.predict(X_ultima)[0]
    probabilidad = modelo.predict_proba(X_ultima)[0][1]  # prob. de clase BUY (1)

    señales_actuales[ticker] = {
        'fecha': ultima_fila['fecha'].iloc[0],
        'señal': 'BUY' if prediccion == 1 else 'SELL',
        'probabilidad_buy': float(probabilidad),
    }

    print(f"  {ticker}: {señales_actuales[ticker]['señal']}  "
          f"(prob. BUY = {probabilidad:.2%}, fecha = {ultima_fila['fecha'].iloc[0].date()})")


  FSM: BUY  (prob. BUY = 50.00%, fecha = 2026-07-02)
  VOLCABC1.LM: BUY  (prob. BUY = 50.59%, fecha = 2026-07-02)
  ABX.TO: SELL  (prob. BUY = 68.52%, fecha = 2026-07-03)
  BVN: BUY  (prob. BUY = 58.13%, fecha = 2026-07-02)
  BHP: BUY  (prob. BUY = 51.92%, fecha = 2026-07-02)


## 9. Guardado en MongoDB (`predicciones` y `metricas_modelos`)

Antes de insertar borramos los documentos previos de estos mismos tickers,
igual que hace el Notebook 1 con `precios_ohlcv`, para evitar duplicados si
el notebook se vuelve a ejecutar.

In [22]:
from datetime import datetime, timezone

tickers_procesados = list(modelos_entrenados.keys())

documentos_predicciones = []
documentos_metricas = []

for ticker in tickers_procesados:
    señal = señales_actuales[ticker]
    documentos_predicciones.append({
        'ticker': ticker,
        'fecha': señal['fecha'].strftime('%Y-%m-%d'),
        'señal': señal['señal'],
        'probabilidad_buy': señal['probabilidad_buy'],
        'created_at': datetime.now(timezone.utc),
    })

    met = metricas_por_ticker[ticker]
    documentos_metricas.append({
        'ticker': ticker,
        'fecha': datetime.now(timezone.utc).strftime('%Y-%m-%d'),
        'modelo': 'SVC',
        'mejores_hiperparametros': modelos_entrenados[ticker].get_params(),
        'accuracy': met['accuracy'],
        'precision': met['precision'],
        'recall': met['recall'],
        'f1': met['f1'],
        'matriz_confusion': met['matriz_confusion'],
        'n_train': len(splits[ticker]['X_train']),
        'n_test': len(splits[ticker]['X_test']),
        'created_at': datetime.now(timezone.utc),
    })

# Borramos predicciones/métricas previas de estos tickers antes de insertar
col_predicciones.delete_many({'ticker': {'$in': tickers_procesados}})
col_metricas.delete_many({'ticker': {'$in': tickers_procesados}})

if documentos_predicciones:
    res_pred = col_predicciones.insert_many(documentos_predicciones)
    print(f"✅ {len(res_pred.inserted_ids)} documentos insertados en 'predicciones'")

if documentos_metricas:
    res_met = col_metricas.insert_many(documentos_metricas)
    print(f"✅ {len(res_met.inserted_ids)} documentos insertados en 'metricas_modelos'")


✅ 5 documentos insertados en 'predicciones'
✅ 5 documentos insertados en 'metricas_modelos'


## 10. Verificación final

In [23]:
import pprint

print("=== Verificación de 'predicciones' ===\n")
for ticker in tickers_procesados:
    doc = col_predicciones.find_one({'ticker': ticker}, sort=[('fecha', -1)])
    estado = "✅" if doc else "⚠️"
    print(f"  {estado} {ticker}: {doc['señal'] if doc else 'sin datos'}")

print("\n=== Verificación de 'metricas_modelos' ===\n")
for ticker in tickers_procesados:
    doc = col_metricas.find_one({'ticker': ticker}, sort=[('fecha', -1)])
    estado = "✅" if doc else "⚠️"
    f1 = f"F1={doc['f1']:.4f}" if doc else "sin datos"
    print(f"  {estado} {ticker}: {f1}")

print("\n--- Ejemplo de documento de predicción ---")
pprint.pprint(col_predicciones.find_one({'ticker': tickers_procesados[0]}, sort=[('fecha', -1)]))

print("\n--- Ejemplo de documento de métricas ---")
pprint.pprint(col_metricas.find_one({'ticker': tickers_procesados[0]}, sort=[('fecha', -1)]))


=== Verificación de 'predicciones' ===

  ✅ FSM: BUY
  ✅ VOLCABC1.LM: BUY
  ✅ ABX.TO: SELL
  ✅ BVN: BUY
  ✅ BHP: BUY

=== Verificación de 'metricas_modelos' ===

  ✅ FSM: F1=0.6333
  ✅ VOLCABC1.LM: F1=0.4889
  ✅ ABX.TO: F1=0.6538
  ✅ BVN: F1=0.5778
  ✅ BHP: F1=0.5116

--- Ejemplo de documento de predicción ---
{'_id': ObjectId('6a483a09ee1c39891044e57e'),
 'created_at': datetime.datetime(2026, 7, 3, 22, 39, 5, 529000),
 'fecha': '2026-07-02',
 'probabilidad_buy': 0.5,
 'señal': 'BUY',
 'ticker': 'FSM'}

--- Ejemplo de documento de métricas ---
{'_id': ObjectId('6a483a0aee1c39891044e583'),
 'accuracy': 0.4634146341463415,
 'created_at': datetime.datetime(2026, 7, 3, 22, 39, 5, 530000),
 'f1': 0.6333333333333333,
 'fecha': '2026-07-03',
 'matriz_confusion': [[0, 22], [0, 19]],
 'mejores_hiperparametros': {'C': 0.1,
                             'break_ties': False,
                             'cache_size': 200,
                             'class_weight': None,
                          